In [3]:
import pyabf
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt, butter, filtfilt, find_peaks
import pywt
from sklearn.decomposition import FastICA
import matplotlib.gridspec as gridspec
import cv2
import networkx as nx

def norm_s(s, g=2000):
    s = s / g
    s = (s - np.mean(s)) / np.std(s)
    return s

def filtro_pasa_bajas(signal, cutoff, fs, order):
    # Diseño del filtro (Butterworth)
    b, a = butter(order, cutoff / (fs / 2), btype='low')
    # Aplicación del filtro (filtrado con mínima distorsión de fase)
    return filtfilt(b, a, signal)

# Cargar el archivo .npy
s = [np.load(f's_007{i}.npy') for i in range(14)]
s = np.array(s)
fs = 500

# Recortar la señal
duracion = 10 * 60 #Segundos
t2 = 600  #Recortar hasta 5 segundos
ln = np.shape(s)[1]  #Longitud de la señal
lm = int(np.floor(ln * t2 / duracion))

s = s[:, :lm]

s_filt = filtro_pasa_bajas(s, 20, fs, 4)
t = np.arange(lm) / fs

n_signals, n_muestras = s_filt.shape

y_max = np.max(s_filt)
y_min = np.min(s_filt)

kernel_vertical = np.array([
    [-50, 0, 50]
])

bordes = cv2.filter2D(s_filt, -1, kernel_vertical)
bordes = cv2.convertScaleAbs(bordes)
_, bordes_bin = cv2.threshold(bordes, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

suma_vertical = np.sum(bordes_bin, axis=0)

# Encuentra los picos con altura entre 1000 y 2500
sincronizaciones, propiedades = find_peaks(suma_vertical, height=(5), distance=45)

sz = 45  # Muestras a cada lado
sz_amp = int(1.35 * sz)  # Muestras a cada lado para la ventana ampliada
t_vent = 2 * sz * 1000 / fs
print(f"Ventana de tiempo: {t_vent:.2f} ms")
print(f"Ventana de tiempo ampliada: {2 * sz_amp * 1000 / fs:.2f} ms")

valid_events = []

for pc in sincronizaciones:
    # Verificar que la ventana alrededor de pc quepa
    if pc - sz >= 0 and pc + sz <= s_filt.shape[1]:
        
        ventana_prov = s_filt[0, pc - sz : pc + sz]  # Tomar ventana en el primer canal
        
        # Buscar el índice relativo del pico máximo absoluto dentro de la ventana
        idx_max_rel = np.argmax(np.abs(ventana_prov))
        
        # Convertir a índice absoluto en la señal completa
        idx_max_abs = (pc - sz) + idx_max_rel
        
        # Verificar que la ventana definitiva alrededor del pico también quepa
        if idx_max_abs - sz >= 0 and idx_max_abs + sz <= s_filt.shape[1]:
            valid_events.append(idx_max_abs)

# Ordenar y eliminar solapamientos
valid_events = sorted(valid_events)

filtered_events = []
last_event = -2 * sz  # Inicializar con valor imposible

for ev in valid_events:
    if ev - last_event >= 2 * sz:
        filtered_events.append(ev)
        last_event = ev

print(f"Número de eventos válidos sin solapamiento: {len(filtered_events)}")

# Crear matrices para las ventanas
s_scr = np.zeros((n_signals, 2 * sz, len(filtered_events)))
s_scr_amp = np.zeros((n_signals, 2 * sz_amp, len(filtered_events)))

t_scr = np.zeros((2 * sz, len(filtered_events)))
t_scr_amp = np.zeros((2 * sz_amp, len(filtered_events)))

for k, pc in enumerate(filtered_events):
    segment = slice(pc - sz, pc + sz)
    segment_amp = slice(pc - sz_amp, pc + sz_amp)
    for i in range(n_signals):
        s_scr[i, :, k] = s_filt[i, segment]
        s_scr_amp[i, :, k] = s_filt[i, segment_amp]
    t_scr[:, k] = t[segment]
    t_scr_amp[:, k] = t[segment_amp]

print(f"Forma de s_scr: {s_scr.shape}, Forma de t_scr: {t_scr.shape}")
print(f"Forma de s_scr_amp: {s_scr_amp.shape}, Forma de t_scr_amp: {t_scr_amp.shape}")

np.save('s_scr_07.npy', s_scr)
np.save('t_scr_07.npy', t_scr)

Ventana de tiempo: 180.00 ms
Ventana de tiempo ampliada: 240.00 ms
Número de eventos válidos sin solapamiento: 1195
Forma de s_scr: (14, 90, 1195), Forma de t_scr: (90, 1195)
Forma de s_scr_amp: (14, 120, 1195), Forma de t_scr_amp: (120, 1195)
